# Analysis of Stiffness Matrix

Below code analyses the stiffness matrix obtained from the respective Kglobal.dat file generated during the solving phase.
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.



To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [4]:
import os
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# 1. INDIVIDUAL FILE ANALYSIS
import os
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh

# 1. SETUP
os.chdir(r"C:\Calculix\ccx_2.23_wsl\Stiffness matrices")
filename = "Kglobal_shelltest.dat"

# 2. LOAD DATA
try:
    data = np.loadtxt(filename)
    i, j, v = data[:, 0].astype(int) - 1, data[:, 1].astype(int) - 1, data[:, 2]
except Exception as e:
    print(f"Error: {e}")
    exit()

# 3. RECONSTRUCT MATRIX
ndof = int(max(i.max(), j.max()) + 1)
K_ut = coo_matrix((v, (i, j)), shape=(ndof, ndof)).tocsr()
K_sparse = K_ut + K_ut.T - coo_matrix((K_ut.diagonal(), (range(ndof), range(ndof))), shape=(ndof, ndof)).tocsr()

# 4. CALCULATE TRACE (Total Matrix Stiffness)
# The trace is the sum of the diagonal entries.
# In FEA, this is a proxy for the total strain energy capacity of the mesh.
matrix_trace = K_sparse.diagonal().sum()

# 5. CALCULATE EIGENVALUES (Physical Modes)
# k=12 to ensure we get past the 6 rigid body modes (0.0 values)
vals, _ = eigsh(K_sparse, k=12, sigma=0, which="LM")
physical_eigenvalues = np.sort(np.abs(vals))[6:] # Discard first 6 (Rigid Body Modes)

print(f"--- Physical Audit for {filename} ---")
print(f"Matrix Size (DOFs): {ndof}")
print(f"Matrix Trace (Total Stiffness): {matrix_trace:.6e}")
print("\nFirst 3 Physical Eigenvalues (Bending/Torsion Modes):")
for idx, val in enumerate(physical_eigenvalues[:3]):
    print(f"  Mode {idx+1}: {val:.6e}")
# ---------------------------------------------------------
# 2. TREND PLOTTING (Scenario 2 Data)
# ---------------------------------------------------------

thickness = np.array([0.10, 0.11, 0.12, 0.15, 0.20])
trace_values = np.array([2.381790e+13, 2.273527e+13, 2.192759e+13, 2.060437e+13, 2.041534e+13])
#condition_numbers = np.array([5.2306e+04, 1.8690e+04, 8.9056e+03, 8.6391e+03, 8.5083e+03])

# Replace these with the lambda_7 values you get from running the analysis on each file
# I've put in a placeholder trend based on typical EI ~ h^3 scaling
lambda_7_values = np.array([4.355317e+05, 4.790462e+05, 5.225510e+05, 6.530021e+05, 8.158316e+05])

fig, axs = plt.subplots(1, 2, figsize=(20, 5))

# Plot 1: Trace (The V-Curve)
axs[0].plot(thickness, trace_values, 'o-', color='firebrick')
axs[0].set_title('Global Trace (Total Energy)', fontweight='bold')
axs[0].set_xlabel('Thickness [m]')
axs[0].set_ylabel('Trace')
axs[0].grid(True, alpha=0.3)

# Plot 2: Condition Number (Numerical Stability)
#axs[1].plot(thickness, condition_numbers, 's-', color='navy')
#axs[1].set_title('Condition Number (Matrix Health)', fontweight='bold')
#axs[1].set_xlabel('Thickness [m]')
#axs[1].set_ylabel('Cond Num (Log Scale)')
#axs[1].set_yscale('log')
#axs[1].grid(True, alpha=0.3)

# Plot 3: 7th Eigenvalue (Fundamental Bending Stiffness)
axs[1].plot(thickness, lambda_7_values, 'd-', color='forestgreen', linewidth=2)
axs[1].set_title('7th Eigenvalue (Bending Mode)', fontweight='bold')
axs[1].set_xlabel('Thickness [m]')
axs[1].set_ylabel('$\lambda_7$ Magnitude')
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



<>:79: SyntaxWarning: invalid escape sequence '\l'
<>:79: SyntaxWarning: invalid escape sequence '\l'
C:\Users\adity\AppData\Local\Temp\ipykernel_16708\2528524732.py:79: SyntaxWarning: invalid escape sequence '\l'
  axs[1].set_ylabel('$\lambda_7$ Magnitude')
C:\Users\adity\AppData\Local\Temp\ipykernel_16708\2528524732.py:79: SyntaxWarning: invalid escape sequence '\l'
  axs[1].set_ylabel('$\lambda_7$ Magnitude')


RuntimeError: Factor is exactly singular